# ⚠️ REQUIRED BEFORE RUNNING — Enable Free GPU

**If you skip this step, the notebook will crash or take 30+ minutes.**

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Set **Hardware accelerator** to **T4 GPU**
4. Click **Save**

Then: **Runtime → Run all** (Ctrl+F9 / Cmd+F9)

---

# TunedAI Labs — Causal Reasoning: 4-Model Comparison

This notebook compares four AI models on the same causal reasoning questions:

| Model | What it is |
|---|---|
| Base Qwen 2.5-7B | An unmodified open-source model |
| GPT-4o | OpenAI's best model (optional — needs API key) |
| Claude 3.5 Sonnet | Anthropic's best model (optional — needs API key) |
| **TunedAI Labs Causal Model** | The same Qwen 2.5-7B, fine-tuned by TunedAI Labs on causal reasoning |

---

## Before you start — one required step

**You must enable a free GPU or the models will load very slowly.**

1. Click **Runtime** in the menu above
2. Click **Change runtime type**
3. Under **Hardware accelerator**, select **T4 GPU**
4. Click **Save**

Then click **Runtime → Run all** (or press Ctrl+F9). The notebook will run automatically.

Loading takes about **2 minutes**. You will see progress messages appear. Scroll down after it finishes to read the results.

---

### About the questions

Every question in this notebook comes from a book or paper published **before AI existed**. The correct answers were worked out by human experts — philosophers, doctors, and statisticians — decades ago. This means the models cannot have simply memorized the answers from the internet.

Sources: John Snow (1854), Ignaz Semmelweis (1847), E.H. Simpson (1951), Bradford Hill (1965), David Hume (1748), R.A. Fisher (1935)
---

### A note on these questions

The six questions below are a representative sample. We tested the TunedAI Labs model across a much wider range of causal reasoning problems — different domains, different structures, different levels of difficulty — and saw consistent results in the same range. The full benchmark score of **96.96% across 10,112 questions** reflects that broader testing. These six are here because they are the most intuitive to read and verify.


---
## Cell 1 of 4 — Install required packages

This installs the software needed to run the models. You will see a lot of text scroll by — that is normal. Wait for it to finish before the next cell runs.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes torch openai anthropic

---
## Cell 2 of 4 — Optional: Add your API keys

If you want to include **GPT-4o** and **Claude 3.5** in the comparison, paste your API keys below between the quotes.

- OpenAI key: get one at **platform.openai.com** → API Keys
- Anthropic key: get one at **console.anthropic.com** → API Keys

**If you leave these blank, those columns will be skipped.** Base Qwen vs TunedAI Labs still runs with no keys needed.

In [ ]:
OPENAI_API_KEY    = ""   # paste your OpenAI key here (optional)
ANTHROPIC_API_KEY = ""   # paste your Anthropic key here (optional)

---
## Cell 3 of 4 — Load the models

This downloads and loads two versions of the same AI model:
- The original unmodified version (Base Qwen 2.5-7B)
- TunedAI Labs fine-tuned version with causal reasoning training

**This takes about 2 minutes.** You will see messages like "Loading tokenizer", "Loading base model", "Loading TunedAI Labs adapter". Wait for the green checkmark ✓ before continuing.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import openai, anthropic

BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER_REPO = "tunedailabs/causal-reasoning-qwen-7b"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("Loading base model (~90 seconds on A100)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    trust_remote_code=True,
)

print("Loading TunedAI Labs causal reasoning adapter...")
tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
tuned_model.eval()

oai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("\n✓ All models ready. Scroll down to see the results.")


---
## Cell 4 of 4 — Helper functions

This sets up the comparison engine. It runs automatically — nothing to change here.

In [ ]:
SYSTEM = "You are a careful reasoner. Answer questions about causation, association, intervention, and counterfactuals precisely and correctly."

def ask_local(question, use_adapter=True, max_new_tokens=350):
    if use_adapter:
        tuned_model.enable_adapter_layers()
    else:
        tuned_model.disable_adapter_layers()
    messages = [{"role":"system","content":SYSTEM},{"role":"user","content":question}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = tuned_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

def ask_gpt4(question):
    if not oai_client:
        return "[Skipped — no OpenAI API key provided]"
    r = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role":"system","content":SYSTEM},{"role":"user","content":question}],
        max_tokens=400
    )
    return r.choices[0].message.content.strip()

def ask_claude(question):
    if not ant_client:
        return "[Skipped — no Anthropic API key provided]"
    r = ant_client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=400,
        system=SYSTEM,
        messages=[{"role":"user","content":question}]
    )
    return r.content[0].text.strip()

def compare_all(question, source="", note=""):
    SEP = "=" * 70
    DIV = "-" * 70
    print(SEP)
    if source: print(f"SOURCE : {source}")
    if note:   print(f"NOTE   : {note}")
    print(SEP)
    print(f"QUESTION:\n{question}\n")
    for label, fn in [
        ("BASE QWEN 2.5-7B (untuned — the starting point)", lambda q: ask_local(q, use_adapter=False)),
        ("GPT-4o",                                          ask_gpt4),
        ("CLAUDE 3.5 SONNET",                              ask_claude),
        ("TUNEDAI CAUSAL MODEL ★",                         lambda q: ask_local(q, use_adapter=True)),
    ]:
        print(DIV)
        print(f"[ {label} ]")
        print(DIV)
        print(fn(question))
        print()

print("✓ Ready — running all tests now...")


---
# Results

Each section below shows all four models answering the same question.

The **BASE QWEN** column is the model before any training on causal reasoning.
The **TUNEDAI** column is the same model after fine-tuning.

---
## Test 1 — Simpson's Paradox
**Source:** E.H. Simpson, *The Interpretation of Interaction in Contingency Tables*, 1951

A real treatment study where one treatment is better for every subgroup of patients — but appears worse in the combined total. Choosing the wrong treatment could harm patients. The correct answer requires understanding why the combined statistic is misleading.

In [ ]:
compare_all(
    question="""Kidney stone treatment study:

Small stones: Treatment A 93% success (81/87 patients), Treatment B 87% success (234/270 patients)
Large stones:  Treatment A 73% success (192/263 patients), Treatment B 69% success (55/80 patients)
Combined:      Treatment A 78% (273/350), Treatment B 83% (289/350)

Treatment A is better for patients with small stones.
Treatment A is better for patients with large stones.
Yet Treatment B appears better in the combined total.

A patient arrives and the doctor does not yet know their stone size.
Which treatment should the doctor give, and why does the combined percentage give the wrong answer?""",
    source="E.H. Simpson, 1951",
    note="Correct answer: administer Treatment A. The combined total is misleading because stone size is a confounder."
)

---
## Test 2 — John Snow and the Cholera Pump
**Source:** John Snow, *On the Mode of Communication of Cholera*, 1854

Snow noticed that households using one water company had far higher cholera death rates. Then he removed the handle from the Broad Street pump and the local outbreak stopped. These are two different types of evidence — and only one of them justifies a causal claim.

In [ ]:
compare_all(
    question="""In 1854, John Snow observed:
- Households using Southwark & Vauxhall water: 315 cholera deaths per 10,000 houses
- Households using Lambeth water: 37 deaths per 10,000 houses

He then removed the handle from the Broad Street water pump.
The cholera outbreak in that district subsided.

Question 1: What does the difference in death rates between the two water companies tell us?
Question 2: What does the pump handle removal tell us that the death rate comparison alone cannot?
Question 3: Why does the order of reasoning matter — starting from a mechanism and then intervening, rather than just observing patterns?""",
    source="John Snow, 1854",
    note="Correct answer: the comparison establishes association. The pump removal establishes causation — it is a real-world intervention."
)

---
## Test 3 — Semmelweis and Handwashing
**Source:** Ignaz Semmelweis, Vienna General Hospital records, 1847

Semmelweis noticed that the ward staffed by doctors (who performed autopsies) had dramatically higher death rates than the ward staffed by midwives. He then mandated handwashing and death rates collapsed. This is a before-and-after intervention.

In [ ]:
compare_all(
    question="""In 1847, Semmelweis observed at Vienna General Hospital:
- Doctors' ward (doctors went from performing autopsies to delivering babies): 10-35% death rate from childbed fever
- Midwives' ward (midwives did not perform autopsies): 1-2% death rate

Semmelweis then required all doctors to wash their hands with chlorinated lime solution before entering the maternity ward.
The death rate in the doctors' ward immediately fell below 2%.

Before the handwashing requirement: could Semmelweis conclude that performing autopsies caused the deaths?
After the handwashing requirement: what can he now conclude that he could not before?
Explain the difference between what the observation tells us and what the intervention tells us.""",
    source="Ignaz Semmelweis, 1847",
    note="Correct answer: observation establishes association. Intervention (handwashing mandate) establishes that the transmission pathway could be broken — a causal claim."
)

---
## Test 4 — Bradford Hill and Smoking
**Source:** Bradford Hill, *The Environment and Disease: Association or Causation?*, 1965

No experiment was ever run on smoking — you cannot randomly assign people to smoke for decades. Critics argued that without an experiment, you can only say smoking is correlated with lung cancer, not that it causes it. Bradford Hill proposed a way to reason about causation from observation alone.

In [ ]:
compare_all(
    question="""By 1965, scientists had observed:
- Smokers get lung cancer at 9-10 times the rate of non-smokers
- This finding had been replicated across many countries and decades
- Heavier smokers get cancer at higher rates than lighter smokers
- Lung cancer is diagnosed years after someone starts smoking, never before
- Cigarette smoke contains known carcinogens

No randomized controlled trial was conducted. You cannot force people to smoke for decades.
Critics argued: without an experiment, this is only a correlation.

Can we conclude that smoking causes lung cancer from this evidence?
What is the strongest argument that we can?
What is the strongest argument that we cannot?""",
    source="Bradford Hill, 1965",
    note="Correct answer: yes, the totality of evidence crosses the threshold for causal inference. Temporality and dose-response are particularly powerful."
)

---
## Test 5 — Hume's Billiard Balls (and Where the Logic Breaks)
**Source:** David Hume, *An Enquiry Concerning Human Understanding*, 1748

Hume proposed a simple test for causation: if the first thing had not happened, the second thing would not have happened. This works for simple cases but breaks down in a specific type of situation. Philosophers spent 250 years trying to fix it.

In [ ]:
compare_all(
    question="""Hume (1748) proposed: to find the cause of something, ask — if the first event had NOT happened, would the second event still have happened? If not, the first event is the cause.

Apply this test to two cases:

Case 1 — Simple: A driver runs a red light and strikes a pedestrian who is injured.
If the driver had NOT run the red light, would the pedestrian have been injured? No.
So by Hume's test: the driver running the red light caused the injury. ✓

Case 2 — Harder: Two people independently decide to poison the same victim's drink.
Person A adds a lethal dose of poison. Person B (not knowing this) also adds a lethal dose.
The victim drinks and dies.

Apply Hume's test:
- If Person A had NOT added poison: the victim still dies (from B's poison). So A did NOT cause the death?
- If Person B had NOT added poison: the victim still dies (from A's poison). So B did NOT cause the death?

Does Hume's test give the right answer for Case 2? If not, what does this reveal about the test's limits?""",
    source="David Hume, 1748",
    note="Correct answer: Hume's test fails for Case 2. Both people are causes, but neither passes the but-for test. This is the problem of overdetermination."
)

---
## Test 6 — Fisher and Randomization
**Source:** R.A. Fisher, *The Design of Experiments*, 1935

The same drug can appear harmful in one study and beneficial in another. Fisher showed that random assignment is what makes the difference — and explained exactly why.

In [ ]:
compare_all(
    question="""Two studies of the exact same drug:

Study A — Observational: Doctors choose which patients get the drug.
Doctors naturally give the drug to sicker patients (they need it more).
Result: The drug group has WORSE outcomes than the no-drug group.

Study B — Randomized (Fisher's method): A coin flip decides who gets the drug.
Result: The drug group has BETTER outcomes than the no-drug group.

Why do these studies give opposite results for the same drug?
What does random assignment do that doctor assignment cannot?
Why does Study B let us say 'the drug works' but Study A does not — even if Study A had 100 times more patients?""",
    source="R.A. Fisher, 1935",
    note="Correct answer: doctor assignment confounds sickness with treatment. Random assignment makes the two groups identical on all factors, so any difference must be due to the drug."
)

---
## Test 7 — Snow's Natural Experiment
**Source:** John Snow, *On the Mode of Communication of Cholera*, 1854

This is a different observation from the pump handle — and arguably stronger. Two water companies served the same streets, with pipes running side by side. Households had no way to choose which company supplied them.

In [ ]:
compare_all(
    question="""In 1854, two water companies supplied houses on the same streets in London — pipes running side by side, households having no say in which company served them:

Southwark & Vauxhall Company: drew water from the Thames below all sewage outfalls.
Deaths per 10,000 houses: 315

Lambeth Company: drew water from the Thames above the city, upstream of outfalls.
Deaths per 10,000 houses: 37

The rest of London: 59 deaths per 10,000.

Why is the phrase 'households had no say in which company served them' the key detail that makes this more than a correlation?
What does this natural experiment establish that Snow could not have established by comparing neighborhoods?
Why is this stronger evidence for causation than simply observing that cholera was more common near the river?""",
    source="John Snow, 1854",
    note="Correct answer: random-like assignment of supplier removes self-selection and neighborhood confounding. It approximates an experiment without intervention."
)

---
## Test 8 — Mill's Method of Difference
**Source:** John Stuart Mill, *A System of Logic*, 1843

Mill identified a set of rules for discovering causes from observation alone. The Method of Difference is the most powerful: find two situations identical in every way except one, and that one difference is the cause.

In [ ]:
compare_all(
    question="""Two villages in England, 1845. Investigators confirm they are identical in every measurable way: same diet, same water source, same altitude, same occupations, same population density, same climate.

One difference: Village A switched from wooden water pipes to lead pipes in 1839. Village B kept its wooden pipes.

Over the next decade:
- Village A: rising cases of severe abdominal pain, muscle weakness, and partial paralysis among residents
- Village B: no such cases

Mill's Method of Difference says: if two situations are alike in every respect except one, and a different outcome occurs, that one difference is the cause.

Apply this method: what can we conclude?
What would need to be true of the villages for this conclusion to hold?
What would it take to falsify the conclusion?""",
    source="John Stuart Mill, A System of Logic, 1843",
    note="Correct answer: lead pipes caused the illness. The method requires the situations be truly identical on all other factors. A third unmeasured difference would break the conclusion."
)

---
## Test 9 — Nightingale and the Confounded Mortality Rate
**Source:** Florence Nightingale, *Notes on Hospitals*, 1863

Nightingale is remembered for nursing, but her real contribution was statistical. She identified how crude mortality rates mislead — and built the visual tools to prove it.

In [ ]:
compare_all(
    question="""Hospital mortality statistics in England, 1861:

London teaching hospitals: 90.8 deaths per 1,000 patients
Rural cottage hospitals: 30.2 deaths per 1,000 patients

A journalist writes: 'City hospitals are three times more deadly than rural ones. Patients should avoid London.'

Nightingale's own records show:
- London teaching hospitals admit patients arriving by emergency, in advanced stages of disease, requiring surgery
- Rural cottage hospitals admit mainly minor ailments, childbirth, and recovery cases
- The two hospital types serve almost no patients in common

Question 1: Why is the journalist's causal claim wrong?
Question 2: What variable is the journalist ignoring, and what is its role in the causal structure?
Question 3: What comparison would you need to make a fair statement about hospital quality?""",
    source="Florence Nightingale, Notes on Hospitals, 1863",
    note="Correct answer: case severity is a confounder. The journalist treats hospital type as the cause of death, ignoring that patients arrive in worse condition. A fair comparison would compare outcomes for the same severity level."
)

---
## Test 10 — Goldberger and Pellagra
**Source:** Joseph Goldberger, U.S. Public Health Service reports, 1915–1926

Pellagra was killing tens of thousands of Americans in the South. The established view was that it was contagious — caused by a germ. Goldberger designed a series of experiments to test this, including injecting himself with material from pellagra patients.

In [ ]:
compare_all(
    question="""By 1915, pellagra killed tens of thousands annually in the American South. The leading theory: it was an infectious disease caused by a germ.

Goldberger ran three sequential studies:

Study 1 — Diet improvement: Goldberger gave a Mississippi orphanage a more varied diet. Pellagra cases dropped from 68 to 0 within months.

Study 2 — Diet induction: Goldberger fed Mississippi prisoners a corn-heavy diet (their normal prison fare) for 5 months. 6 of 11 developed pellagra.

Study 3 — The filth party: Goldberger and 16 volunteers (including his wife) injected themselves with blood, swabbed their nasal passages with secretions, and swallowed capsules containing skin scrapings from active pellagra patients. None developed pellagra.

Before Study 3: what could Goldberger conclude from Studies 1 and 2?
After Study 3: what additional claim can he now make?
Why was Study 3 necessary even after Studies 1 and 2 produced such clear results?""",
    source="Joseph Goldberger, U.S. Public Health Service, 1915",
    note="Correct answer: Studies 1-2 show diet causes/prevents pellagra. Study 3 rules out the competing hypothesis (infection). Without ruling out the germ theory, skeptics could say the diet change eliminated something that spread the germ."
)

---
## Test 11 — Lind's Scurvy Trial
**Source:** James Lind, *A Treatise of the Scurvy*, 1753

Lind conducted what is considered the first controlled clinical trial in history. Twelve sailors. Six treatments. Two weeks. One clear result. The British Navy rejected his findings for 42 years.

In [ ]:
compare_all(
    question="""1747, aboard HMS Salisbury. Twelve sailors with advanced scurvy. Lind assigned them in pairs to six different daily supplements, all eating the same base diet:

Pair 1: One quart of cider per day
Pair 2: 25 drops of sulfuric acid
Pair 3: Six spoonfuls of vinegar
Pair 4: Half a pint of seawater
Pair 5: Two oranges and one lemon (until supply ran out after 6 days)
Pair 6: A paste of garlic, mustard, and radish

After two weeks, Pair 5 had fully recovered and returned to duty. No other pair showed meaningful improvement.

Question 1: What makes this more than an anecdote, despite having only 2 patients per treatment?
Question 2: Why did Lind assign the same base diet to all 12 sailors — what would go wrong without this?
Question 3: The Navy rejected his findings for 42 years and continued losing sailors to scurvy. What additional evidence would have made rejection harder?""",
    source="James Lind, A Treatise of the Scurvy, 1753",
    note="Correct answer: simultaneous comparison controls for time and base diet. The base diet controls for confounders. Rejection would have been harder with replication, a larger trial, or understanding of why citrus works (mechanism)."
)

---
## Test 12 — Koch's Postulates and the Logic of Causal Proof
**Source:** Robert Koch, tuberculosis research, 1876–1884

Koch identified four steps required to prove that a specific microorganism causes a specific disease. Each step eliminates a specific type of causal error. Skipping any one of them leaves a logical gap.

In [ ]:
compare_all(
    question="""Robert Koch proposed four requirements to prove a microorganism causes a disease:

Step 1 — The organism must be found in every sick individual (and ideally absent in healthy ones)
Step 2 — The organism must be isolated from the sick individual and grown in pure culture outside the body
Step 3 — The cultured organism must be introduced to a healthy individual and produce the same disease
Step 4 — The organism must be re-isolated from the newly sick individual and shown to be identical to the original

Koch used this method to prove Mycobacterium tuberculosis causes tuberculosis in 1882.

Question 1: Why isn't Step 1 alone sufficient to conclude the organism causes the disease?
Question 2: What specific causal error does Step 3 rule out that Steps 1 and 2 cannot?
Question 3: What does Step 4 add that Step 3 alone does not prove?
Question 4: Some modern diseases (HIV, some prion diseases) don't satisfy all four steps — what does this reveal about the limits of Koch's framework?""",
    source="Robert Koch, 1876-1884",
    note="Correct answer: Step 1 alone = correlation. Step 2 = isolation (ruling out multiple agents). Step 3 = causation (intervention). Step 4 = identity (confirming same organism). Modern exceptions show the postulates are sufficient but not necessary conditions."
)

---
## Test 13 — Wald and Survivorship Bias
**Source:** Abraham Wald, Statistical Research Group, Columbia University, 1943

The U.S. military asked statistician Abraham Wald to analyze bullet holes on returning planes and recommend where to add armor. His recommendation was the opposite of what the military expected — and exactly right.

In [ ]:
compare_all(
    question=(
        "Engineers examined hundreds of Allied bombers returning from combat. Bullet hole locations:\n"
        "\n"
        "Wings and tail surfaces: many holes\n"
        "Fuselage body: many holes\n"
        "Engines and cockpit: very few holes\n"
        "\n"
        "Military plan: add armor to wings and fuselage (where most holes were).\n"
        "Abraham Wald: add armor to engines and cockpit (where almost no holes were).\n"
        "\n"
        "Question 1: What was Wald's insight? Why should armor go where there are NO holes?\n"
        "Question 2: What type of causal error was the military making?\n"
        "Question 3: The planes shot in the engines and cockpit are not in this dataset. What happened to them?"
    ),
    source="Abraham Wald, 1943",
    note="Correct answer: planes hit in engines/cockpit didn't return — they were shot down. Armoring the areas that can sustain damage and still fly is wrong. The missing data is the signal."
)

---
## Test 14 — Galton and Regression to the Mean
**Source:** Francis Galton, Regression Towards Mediocrity in Hereditary Stature, 1886

Galton noticed that tall parents had children shorter than themselves, and short parents had children taller. A flight instructor made the same observation and drew exactly the wrong causal conclusion.

In [ ]:
compare_all(
    question=(
        "A flight instructor observes over many years:\n"
        "- After praising a pilot for an unusually GOOD landing, the next landing is typically worse\n"
        "- After criticizing a pilot for an unusually BAD landing, the next landing is typically better\n"
        "\n"
        "Conclusion: praise causes worse performance; criticism causes better performance.\n"
        "\n"
        "Separately, Francis Galton (1886) measured parents and adult children:\n"
        "- Parents in the top 10% for height had children who were tall — but shorter than themselves\n"
        "- Parents in the bottom 10% had children who were short — but taller than themselves\n"
        "\n"
        "Question 1: What is actually causing both the instructor's observation and Galton's data?\n"
        "Question 2: Is the instructor's causal conclusion correct? What would test it properly?\n"
        "Question 3: How is this a trap for anyone using performance data to evaluate whether an intervention worked?"
    ),
    source="Francis Galton, 1886",
    note="Correct answer: regression to the mean. Extreme performances include a luck component; the next measurement reverts toward average regardless of feedback. The instructor is attributing to praise/criticism what is statistical noise."
)

---
## Test 15 — Pasteur's Swan-Neck Flask
**Source:** Louis Pasteur, Memoire sur les corpuscles organises, 1859

The leading scientific debate of the 1850s: does life arise spontaneously from non-living matter, or must it come from existing life? Pasteur settled it with a single elegant intervention.

In [ ]:
compare_all(
    question=(
        "The spontaneous generation debate, 1850s.\n"
        "\n"
        "Critics' argument: boil broth (kills everything), seal it — stays clear. Open it to air — microorganisms appear. "
        "Therefore: something in air spontaneously generates life.\n"
        "\n"
        "Pasteur built a flask with a long curved 'swan-neck' tube — open to air, but the curve traps particles before they reach the broth.\n"
        "Result: broth stayed clear for months, despite being open to air.\n"
        "\n"
        "Then Pasteur tipped the flask so broth touched the dust trapped in the curve.\n"
        "Result: broth became cloudy with microorganisms within days.\n"
        "\n"
        "Question 1: What does tipping the flask tell us that simply sealing it cannot?\n"
        "Question 2: Why was the curved design — open to air but blocking particles — the critical feature?\n"
        "Question 3: What specifically does this experiment rule out about the spontaneous generation theory?"
    ),
    source="Louis Pasteur, 1859",
    note="Correct answer: tipping is an intervention — Pasteur controlled what reached the broth. The curve lets air in but blocks particles, separating 'air' from 'particles in air.' Rules out spontaneous generation from air; proves contamination by particles."
)

---
## Test 16 — Jenner and the First Vaccination
**Source:** Edward Jenner, An Inquiry into the Causes and Effects of the Variolae Vaccinae, 1798

Jenner noticed that milkmaids who caught cowpox never seemed to get smallpox. He tested this with a deliberate intervention on a child — and changed medicine forever.

In [ ]:
compare_all(
    question=(
        "Edward Jenner observed in the English countryside:\n"
        "- Milkmaids frequently caught cowpox (mild disease) from infected cows\n"
        "- During smallpox epidemics, milkmaids almost never caught smallpox\n"
        "- This pattern was consistent across many milkmaids, multiple villages, many years\n"
        "\n"
        "Jenner's hypothesis: cowpox infection prevents smallpox.\n"
        "\n"
        "In 1796, Jenner took material from a cowpox pustule on a milkmaid and scratched it into the arm of James Phipps, an 8-year-old boy. "
        "The boy developed mild cowpox and recovered. "
        "Six weeks later, Jenner scratched material from a smallpox patient into the boy's arm. "
        "The boy did not develop smallpox.\n"
        "\n"
        "Question 1: What does the milkmaid observation alone establish — and what does it not establish?\n"
        "Question 2: What does the deliberate inoculation establish that the observation cannot?\n"
        "Question 3: Jenner tested only one child. Why is this still stronger evidence than observing dozens of milkmaids?"
    ),
    source="Edward Jenner, 1798",
    note="Correct answer: observation = association (milkmaids have cowpox AND no smallpox). Inoculation = Jenner controlled the exposure, ruling out self-selection. One deliberate experiment with controlled exposure > many passive correlations."
)

---
## Test 17 — Reverse Causation
**Source:** Classic problem in causal inference — Hume (1748) and modern epidemiology

Association tells us two things are related. It does not tell us which causes which. This produces some of the most persistent errors in medicine, economics, and public policy.

In [ ]:
compare_all(
    question=(
        "Three observed associations. For each, identify the most likely causal direction and explain how you would test it.\n"
        "\n"
        "Association 1:\n"
        "People admitted to hospitals are sicker on average than people who stay home.\n"
        "A naive observer concludes: hospitals make people sick.\n"
        "\n"
        "Association 2:\n"
        "Countries with more hospitals per capita have higher rates of certain diseases.\n"
        "A policy maker concludes: building more hospitals increases disease.\n"
        "\n"
        "Association 3:\n"
        "Wealthier people are healthier in every country studied.\n"
        "Hypothesis A: money buys health (wealth causes health).\n"
        "Hypothesis B: healthy people earn more (health causes wealth).\n"
        "Hypothesis C: a third factor (childhood environment) causes both.\n"
        "\n"
        "For each association:\n"
        "- What are the possible causal directions?\n"
        "- What evidence would distinguish between them?\n"
        "- What is the most common mistake when interpreting these?"
    ),
    source="Classic causal inference problem",
    note="Correct answer: (1) sick people go to hospitals — selection effect. (2) wealthy areas have more hospitals AND more diagnosis. (3) all three mechanisms likely operate; need longitudinal data or natural experiments to decompose them."
)

---
## Test 18 — Farr's Miasma vs Snow's Water
**Source:** William Farr, Letter to the Registrar General, 1852; John Snow, 1854

William Farr was the leading statistician of the 1850s. He found a perfect correlation between altitude and cholera deaths — and drew exactly the wrong causal conclusion. His data was real. His conclusion was wrong.

In [ ]:
compare_all(
    question=(
        "William Farr, 1852. Analyzing London mortality data:\n"
        "\n"
        "Altitude 0-20 feet above sea level:   102 cholera deaths per 10,000\n"
        "Altitude 20-40 feet:                   65 deaths per 10,000\n"
        "Altitude 40-60 feet:                   34 deaths per 10,000\n"
        "Altitude 60-80 feet:                   27 deaths per 10,000\n"
        "Above 100 feet:                        17 deaths per 10,000\n"
        "\n"
        "Farr concluded: low altitude causes cholera — miasmatic bad air accumulates near the ground.\n"
        "\n"
        "John Snow then showed: low-lying areas were served by Southwark and Vauxhall water (drawn from below sewage outfalls). "
        "High-lying areas were served by Lambeth water (drawn from above). "
        "The altitude correlation was real — but altitude was not the cause.\n"
        "\n"
        "Question 1: Was Farr's data wrong? Was his inference wrong? Explain the difference.\n"
        "Question 2: What type of variable is altitude in this causal story?\n"
        "Question 3: What is the general lesson about finding a variable that perfectly predicts an outcome?"
    ),
    source="William Farr, 1852; John Snow, 1854",
    note="Correct answer: Farr's data was correct; his inference was wrong. Altitude is a proxy/confounder — correlated with water source, which was the true cause. A perfect predictor is not necessarily a cause."
)

---
## Test 19 — Placebo Control and What We Are Actually Measuring
**Source:** Henry Beecher, The Powerful Placebo, JAMA, 1955

In 1955, Beecher reviewed 15 clinical trials. On average, 35% of patients improved when given a sugar pill. This changed the design of every drug trial that followed.

In [ ]:
compare_all(
    question=(
        "Henry Beecher, 1955. Analysis of 15 drug trials, 1,082 patients:\n"
        "- Average placebo response: 35% of patients improved from sugar pills alone\n"
        "- In some conditions (pain, anxiety, nausea): placebo response exceeded 50%\n"
        "- Effect was consistent across different doctors, hospitals, and countries\n"
        "\n"
        "A drug company tests a new painkiller:\n"
        "- 200 patients receive the drug: 65% report significant pain reduction\n"
        "- 200 patients receive nothing (no treatment): 40% report significant pain reduction\n"
        "\n"
        "A critic says: you need a placebo group (sugar pill), not a no-treatment group.\n"
        "\n"
        "Question 1: Why does the critic want a placebo group rather than a no-treatment group?\n"
        "Question 2: The drug produces 65% improvement; no-treatment produces 40%. Does this mean the drug works?\n"
        "Question 3: What exactly is being measured in a trial with a placebo control vs one without?"
    ),
    source="Henry Beecher, JAMA, 1955",
    note="Correct answer: placebo controls for the act of receiving treatment (expectation, attention, ritual). Without it, you measure drug effect + placebo effect combined. The no-treatment comparison doesn't isolate the pharmacological effect."
)

---
## Test 20 — The Streptomycin Trial
**Source:** MRC Streptomycin in Tuberculosis Trials Committee, British Medical Journal, 1948

The first properly randomized controlled trial in medicine. Before 1948, doctors observed TB patients on streptomycin and believed it worked. Bradford Hill insisted on randomization anyway — and was right to.

In [ ]:
compare_all(
    question=(
        "1947. Streptomycin was being used on TB patients. Doctors observing their patients reported improvement "
        "and believed the drug worked. Bradford Hill proposed a randomized trial.\n"
        "\n"
        "107 TB patients enrolled. Random allocation (sealed envelopes):\n"
        "- 55 to streptomycin + bed rest\n"
        "- 52 to bed rest only\n"
        "\n"
        "Results after 6 months:\n"
        "- Streptomycin group: 7% died, 51% showed radiological improvement\n"
        "- Bed rest only: 27% died, 8% showed improvement\n"
        "\n"
        "Before the trial, doctors believed streptomycin worked based on their own patient observations.\n"
        "\n"
        "Question 1: Why was doctor observation of their own patients insufficient to establish that streptomycin caused improvement?\n"
        "Question 2: Name three things that could explain why observed patients improved — things randomization controls for.\n"
        "Question 3: The streptomycin supply was limited — only enough for half the patients. Why did this make the trial more ethical, not less?"
    ),
    source="MRC Streptomycin Trials Committee, BMJ, 1948",
    note="Correct answer: observation confounds patient selection (doctors may treat milder cases), natural disease course, and the placebo effect. Randomization controls all three simultaneously. Limited supply means withholding from some was unavoidable — randomizing was the fairest allocation."
)

---
## Test 21 — Overdetermination and Multiple Sufficient Causes
**Source:** Hart and Honore, Causation in the Law, 1959; philosophy of causation

A single effect can have multiple independent causes, each sufficient to produce it alone. This breaks the standard test for causation — with serious consequences for law, medicine, and policy.

In [ ]:
compare_all(
    question=(
        "Two scenarios of overdetermination:\n"
        "\n"
        "Scenario A — Simultaneous sufficient causes:\n"
        "Two forest fires start independently and simultaneously on opposite sides of a valley. "
        "Each fire alone would destroy the town. The fires merge and destroy the town.\n"
        "\n"
        "Applying Hume's but-for test:\n"
        "- But for Fire 1: the town still burns (Fire 2 would have done it). So Fire 1 did NOT cause the destruction?\n"
        "- But for Fire 2: the town still burns (Fire 1 would have done it). So Fire 2 did NOT cause the destruction?\n"
        "\n"
        "Scenario B — Preemptive causation:\n"
        "A hiker is given a poisoned canteen. Before the hiker drinks, a second person shoots holes in the canteen. "
        "The hiker dies of thirst.\n"
        "\n"
        "Question 1: Does Hume's but-for test give the right answer in Scenario A? If not, what went wrong?\n"
        "Question 2: In Scenario B, who caused the death — the poisoner, the shooter, or both? Apply the but-for test to each.\n"
        "Question 3: What does this reveal about the difference between scientific causation and legal causation?"
    ),
    source="Hart and Honore, Causation in the Law, 1959",
    note="Correct answer: but-for fails for overdetermination — both fires are causes but neither passes the test. In Scenario B, the shooter caused death (poison was never delivered); the poisoner attempted but did not cause it. Science and law can give different verdicts."
)

---
## Test 22 — Doll and Hill's Prospective Doctor Study
**Source:** Richard Doll and Bradford Hill, British Medical Journal, 1954

After their 1950 case-control study linked smoking to lung cancer, critics argued the data could be explained by recall bias. Doll and Hill responded with a prospective study of 40,000 British doctors.

In [ ]:
compare_all(
    question=(
        "1951. Doll and Hill mailed questionnaires to 40,000 British doctors asking about smoking habits. "
        "They tracked all deaths over the following years.\n"
        "\n"
        "Results after 4 years:\n"
        "Lung cancer deaths per 1,000 per year:\n"
        "- Non-smokers:              0.00\n"
        "- Light smokers (1-14/day): 0.47\n"
        "- Moderate (15-24/day):     0.86\n"
        "- Heavy (25+/day):          1.66\n"
        "\n"
        "Additionally: doctors who had quit smoking had lower rates than current smokers.\n"
        "\n"
        "Their 1950 study was retrospective — asking lung cancer patients and healthy controls to remember past smoking.\n"
        "\n"
        "Question 1: What specific criticism of the 1950 study does the prospective design rule out?\n"
        "Question 2: What does the dose-response gradient add to the causal argument?\n"
        "Question 3: What does the 'quitters have lower rates' finding add that neither the retrospective study nor the dose-response alone provides?"
    ),
    source="Doll and Hill, BMJ, 1954",
    note="Correct answer: prospective design eliminates recall bias. Dose-response shows gradient (Bradford Hill criterion). Quitters' lower rates show the effect is reversible — dramatically strengthening the causal claim because it demonstrates the cause can be removed."
)

---
## Test 23 — The Berkeley Admissions Paradox
**Source:** P.J. Bickel, E.A. Hammel, J.W. O'Connell, Science, 1975

In 1973, UC Berkeley was sued for sex discrimination in graduate admissions. The aggregate data looked damning. When Bickel examined department-level data, the story reversed completely.

In [ ]:
compare_all(
    question=(
        "UC Berkeley graduate admissions, Fall 1973:\n"
        "\n"
        "Aggregate data:\n"
        "- Male applicants:   8,442 applied, 44% admitted\n"
        "- Female applicants: 4,321 applied, 35% admitted\n"
        "This appears to show a 9-point gap favoring men.\n"
        "\n"
        "Department-level breakdown (top 6 departments by volume):\n"
        "Dept A: Men 62%, Women 82% admitted\n"
        "Dept B: Men 63%, Women 68% admitted\n"
        "Dept C: Men 37%, Women 34% admitted\n"
        "Dept D: Men 33%, Women 35% admitted\n"
        "Dept E: Men 28%, Women 24% admitted\n"
        "Dept F: Men 6%,  Women 7%  admitted\n"
        "\n"
        "In 4 of 6 departments, women are admitted at higher rates.\n"
        "\n"
        "Question 1: How is it possible that women are admitted at lower rates overall but higher rates in most departments?\n"
        "Question 2: What variable explains the paradox and what role does it play in the causal structure?\n"
        "Question 3: If you were advising the university on whether discrimination occurred, which data matters — aggregate or department level? Why?"
    ),
    source="Bickel, Hammel, O'Connell, Science, 1975",
    note="Correct answer: women applied in higher proportions to competitive departments (low admission rates). Men applied more to easier departments. The aggregate doesn't control for department selectivity — a confounder. Department-level data is the correct level of analysis."
)

---
## Test 24 — Mill's Method of Concomitant Variation
**Source:** John Stuart Mill, A System of Logic, 1843

Mill's fifth method of causal discovery: when two things vary together — more of one produces more of the other — this is evidence of a causal link. It captures what we now call dose-response.

In [ ]:
compare_all(
    question=(
        "Workers in a chemical factory are exposed to benzene at varying levels depending on job role.\n"
        "\n"
        "Annual benzene exposure vs leukemia incidence (per 100,000 workers per year):\n"
        "Unexposed (office workers):           6 cases\n"
        "Low exposure (1-10 ppm):             14 cases\n"
        "Medium exposure (10-50 ppm):         31 cases\n"
        "High exposure (50-200 ppm):          58 cases\n"
        "Very high exposure (200+ ppm):      105 cases\n"
        "\n"
        "Each group is otherwise similar in age, smoking rates, and other measured factors.\n"
        "\n"
        "Mill: 'Whatever phenomenon varies whenever another phenomenon varies in some particular manner, "
        "is either a cause or an effect of that phenomenon.'\n"
        "\n"
        "Question 1: What does the gradient tell us that a simple comparison of exposed vs unexposed cannot?\n"
        "Question 2: Mill says this is 'either a cause or an effect.' What would it mean for benzene to be the effect? How would you rule this out?\n"
        "Question 3: What additional evidence would be needed to move from strong causal suspicion to confident causal claim?"
    ),
    source="John Stuart Mill, A System of Logic, 1843",
    note="Correct answer: gradient rules out threshold effects and strengthens causal argument. Benzene as effect makes no sense — you cannot develop leukemia and then absorb more benzene. Mechanism (DNA damage) + animal data + intervention (reduce exposure, measure outcome) complete the case."
)

---
## Test 25 — Snow's Negative Control: The Brewery Workers
**Source:** John Snow, On the Mode of Communication of Cholera, 2nd edition, 1855

During the 1854 Broad Street outbreak, Snow documented not just who died — but who did not. A group of workers lived and worked directly next to the contaminated pump and had zero deaths.

In [ ]:
compare_all(
    question=(
        "The Broad Street cholera outbreak, 1854.\n"
        "\n"
        "At Broad Street stood a large brewery employing over 70 workers who lived on the premises during the outbreak.\n"
        "\n"
        "Deaths on surrounding streets: 500+ within 10 days.\n"
        "Deaths among brewery workers: 0.\n"
        "\n"
        "Investigation: the brewery had its own well. Workers were also given a daily ration of malt liquor. "
        "None drank from the Broad Street pump.\n"
        "\n"
        "Question 1: What does the brewery workers' zero deaths tell Snow that the mortality map alone cannot?\n"
        "Question 2: This group is called a 'negative control.' What does a negative control add to a causal argument?\n"
        "Question 3: If two brewery workers had died of cholera during the outbreak, what would Snow need to investigate to preserve his theory? What finding would falsify it?"
    ),
    source="John Snow, 1855",
    note="Correct answer: negative control shows the proximity to the pump does not cause death — it is drinking from the pump that matters. This distinguishes water from neighborhood, smell, or air as the cause. Falsification: if those two deaths also drank only brewery water, Snow's theory fails."
)

---
## Test 26 — Confounder vs Effect Modifier
**Source:** Epidemiology methodology — Rothman, Modern Epidemiology

Two variables can influence a study in completely different ways. A confounder distorts the apparent relationship. An effect modifier genuinely changes the size of the causal effect. The distinction has major consequences for policy.

In [ ]:
compare_all(
    question=(
        "A clinical trial tests a new blood pressure drug.\n"
        "\n"
        "Scenario A — Confounding:\n"
        "Overall result: drug appears to reduce blood pressure by 5 mmHg.\n"
        "Breakdown: under-50 patients: drug reduces BP by 12 mmHg; over-50: reduces by 11 mmHg.\n"
        "But: older patients were assigned to the drug at higher rates because they had higher baseline BP.\n"
        "When you control for age: the apparent 5 mmHg effect disappears. Age was distorting the result.\n"
        "\n"
        "Scenario B — Effect Modification:\n"
        "Overall result: drug reduces BP by 8 mmHg on average.\n"
        "Breakdown by sex: men: 14 mmHg reduction; women: 2 mmHg reduction.\n"
        "When you control for sex: the overall average stays at 8 mmHg — it is just a mix of two real effects.\n"
        "\n"
        "Question 1: In Scenario A, is age distorting the result or genuinely changing the true causal effect?\n"
        "Question 2: In Scenario B, is sex distorting the result or genuinely changing the true causal effect?\n"
        "Question 3: What is the policy implication of confounding vs effect modification — what action does each call for?"
    ),
    source="Epidemiological methods — Rothman, Greenland, Lash",
    note="Correct answer: A = confounding (the 5 mmHg was wrong — age distorted it). B = effect modification (the 8 mmHg average is technically correct but misleading — the drug should be prescribed to men only). Different causal structures demand different responses."
)

---
## Test 27 — Cochrane and the Coronary Care Paradox
**Source:** Archie Cochrane, Effectiveness and Efficiency: Random Reflections on Health Services, 1972

Doctors in the 1960s believed heart attack patients needed extended hospital bed rest. Cochrane ran a randomized trial to test this. The results shocked the medical establishment — and revealed how badly observation had misled them.

In [ ]:
compare_all(
    question=(
        "1960s. Standard care: heart attack patients spend 3-4 weeks in a coronary care unit under continuous monitoring.\n"
        "Doctors observed their patients and believed this was life-saving. Patients who left early seemed to do worse.\n"
        "\n"
        "Cochrane proposed a randomized trial: half of recovering patients sent home after one week; half stay in hospital.\n"
        "\n"
        "Interim results after the first group:\n"
        "- Home care group:     6 deaths per 100 patients\n"
        "- Hospital care group: 8 deaths per 100 patients\n"
        "\n"
        "When Cochrane showed these interim results, the cardiologists demanded he stop the trial and send all patients home immediately — on ethical grounds.\n"
        "\n"
        "Question 1: Why did doctors' observations of their own patients mislead them into thinking hospital care was better?\n"
        "Question 2: What specific bias caused patients who left the hospital early to 'seem to do worse'?\n"
        "Question 3: The cardiologists first wanted to stop the trial to keep patients in hospital. After seeing interim data, they wanted to stop to send everyone home. What should Cochrane do, and why?"
    ),
    source="Archie Cochrane, 1972",
    note="Correct answer: patients who left early included those who insisted on leaving against advice (non-compliant or sicker). Observation confounds discharge decision with health status. Cochrane should continue — interim results from a small first group are noisy and the trial was designed for a larger sample."
)

---
## Test 28 — Graunt and the Geography of Plague
**Source:** John Graunt, Natural and Political Observations on the Bills of Mortality, 1662

John Graunt was a cloth merchant who became the first demographic statistician. His 1662 analysis of London death records was the first attempt to reason causally about disease from population data.

In [ ]:
compare_all(
    question=(
        "John Graunt, 1665. Analyzing London's Bills of Mortality during plague outbreaks:\n"
        "\n"
        "- Deaths in central city parishes increased 10-20 fold within weeks during outbreaks\n"
        "- Deaths in outer parishes increased far less, or not at all\n"
        "- Plague deaths moved from parish to parish over successive weeks — spreading outward\n"
        "- Wealthy families who fled London early in outbreaks had lower death rates than those who stayed\n"
        "\n"
        "Two competing theories in 1665:\n"
        "Theory A — Miasma: plague arises from corrupted air in crowded, dirty areas. Outskirts safer because air is cleaner.\n"
        "Theory B — Contagion: plague spreads from person to person. It moves outward as infected people or goods travel.\n"
        "\n"
        "Question 1: What does the pattern of plague spreading from center outward over time tell us about which theory is more likely?\n"
        "Question 2: Wealthy families who fled early had better outcomes. Is this evidence for miasma, contagion, or both?\n"
        "Question 3: What single observation would most strongly distinguish between the two theories?"
    ),
    source="John Graunt, Natural and Political Observations on the Bills of Mortality, 1662",
    note="Correct answer: sequential spread outward from a center fits contagion (the agent travels). Miasma predicts simultaneous onset wherever air is bad. Wealthy fleeing fits both (better air elsewhere vs less contact). Decisive test: do people who flee before exposure survive? If yes, contagion. Miasma predicts destination matters, not timing."
)

---
## Test 29 — The Hawthorne Effect
**Source:** Elton Mayo et al., Hawthorne Works studies, Western Electric Company, 1924-1932

Researchers at a factory tested the effect of lighting, rest breaks, and work hours on productivity. Nearly every change they made — including reversals — improved productivity. The intervention was not what they thought it was.

In [ ]:
compare_all(
    question=(
        "Hawthorne Works, Illinois. Researchers studied how working conditions affect productivity.\n"
        "\n"
        "Lighting study:\n"
        "- Increased lighting: productivity increased\n"
        "- Decreased lighting back to original: productivity still increased\n"
        "- Decreased lighting below original: productivity still increased\n"
        "\n"
        "Rest break study:\n"
        "- Added two 5-minute breaks: productivity increased\n"
        "- Extended breaks to 10 minutes: productivity increased\n"
        "- Removed all breaks: productivity remained elevated above baseline\n"
        "\n"
        "The researchers noted: workers knew they were being studied.\n"
        "\n"
        "Question 1: What was actually causing productivity to increase in every condition — including reversals?\n"
        "Question 2: This reveals a fundamental problem with unblinded observation studies. What is it?\n"
        "Question 3: How should a study be designed if you want to measure the true causal effect of lighting on productivity, without this contamination?"
    ),
    source="Elton Mayo, Hawthorne Works, 1924-1932",
    note="Correct answer: being observed changed behavior — workers performed better when watched. The intervention was not lighting or breaks; it was attention. Unblinded studies confound the treatment with the act of treatment. Solution: blind participants to which condition they're in, or use equally-observed controls."
)

---
## Test 30 — Income, Health, and Causal Direction
**Source:** Michael Marmot, Whitehall Study, The Lancet, 1978

Wealthier people are healthier in every country ever studied. The relationship is a smooth gradient — not just poverty vs wealth. But knowing two things are correlated does not tell you which causes which.

In [ ]:
compare_all(
    question=(
        "The Whitehall Study (Marmot, 1978): 17,500 British civil servants, tracked 10 years.\n"
        "\n"
        "Mortality by employment grade:\n"
        "- Senior administrators:    lowest mortality (baseline)\n"
        "- Executive grade:           1.6x higher than senior\n"
        "- Clerical staff:            2.2x higher\n"
        "- Lowest grade workers:      3.5x higher\n"
        "\n"
        "Every step down the hierarchy = worse health. All participants had stable employment and the same national health service.\n"
        "\n"
        "Three competing causal hypotheses:\n"
        "Hypothesis 1 — Wealth causes health: higher income allows better nutrition, less stress, better housing.\n"
        "Hypothesis 2 — Health causes wealth: healthier people are more productive, advance further, earn more.\n"
        "Hypothesis 3 — Common cause: childhood environment or genetics causes both health and career attainment.\n"
        "\n"
        "Question 1: What type of study would distinguish Hypothesis 1 from Hypothesis 2?\n"
        "Question 2: What type of study would detect a common cause (Hypothesis 3) driving both?\n"
        "Question 3: If all three hypotheses are partially true, what does that mean for a policy of increasing income for low-wage workers — will it improve their health?"
    ),
    source="Michael Marmot, Whitehall Study, The Lancet, 1978",
    note="Correct answer: longitudinal studies (measure health before income changes) separate 1 from 2. Twin and sibling studies control for shared genetics and childhood environment. If Hypothesis 3 dominates, income transfers may not improve health as much as early childhood investment."
)

---
## Try Your Own Question

Replace the text below with any causal reasoning question. Examples to try:
- "Does wearing a seatbelt cause people to drive more recklessly?"
- "If a rooster crows before sunrise every day, does the rooster cause the sun to rise?"
- "A city adds more police and crime goes up. Does adding police cause more crime?"

In [ ]:
MY_QUESTION = """
Type your question here and run this cell.
"""

compare_all(MY_QUESTION, source="Custom")

---
## Share What You Saw

Open a [GitHub Issue](https://github.com/tunedailabs/tunedailabs/issues/new) and paste your results.

Tell us which questions the TunedAI Labs model got right that the others did not — or anything surprising you found.

We read every submission.

---

**TunedAI Labs** — We fine-tune open-source LLMs for real-world reasoning.

[tunedailabs.com](https://tunedailabs.com)